## Extracao dos dados

In [ ]:
import json
import subprocess
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, roc_curve, roc_auc_score
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, cross_val_predict, GridSearchCV
)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

SEED = 42
np.random.seed(SEED)

## Funcoes do pipeline

In [ ]:
def encontrar_raiz_do_projeto(inicio):
    """Procura a raiz do projeto para nao depender de caminho fixo."""
    raiz = Path(inicio).resolve()
    while not (raiz / "pyproject.toml").exists():
        if raiz.parent == raiz:
            raise FileNotFoundError("Nao encontrei o pyproject.toml. Abra o notebook dentro do repositorio.")
        raiz = raiz.parent
    return raiz

def carregar_base(caminho, separador=","):
    return pd.read_csv(caminho, sep=separador)

def tratar_dados(dataframe_inicial, colunas_pt_br, colunas_addons):
    df = dataframe_inicial.rename(columns=colunas_pt_br)
    df["target"] = df["target"].map({"Yes": 1, "No": 0})

    # removendo identificador e features sensiveis
    df = df.drop(columns=["id_cliente", "genero", "dependentes"])

    # binarizando features deterministicas
    for coluna in ["casado", "fatura_sem_papel", "possui_servico_telefonico"]:
        df[coluna] = df[coluna].map({"Yes": 1, "No": 0})

    # binarias nao deterministicas
    for coluna in colunas_addons:
        df[coluna] = df[coluna].map({"Yes": 1, "No": 0, "No internet service": 0})
    df["possui_multiplas_linhas"] = df["possui_multiplas_linhas"].map({"Yes": 1, "No": 0, "No phone service": 0})
    df["tipo_contrato"] = df["tipo_contrato"].map({"Month-to-month": 0, "One year": 1, "Two year": 2})

    # one hot
    servico_de_internet_dummies = pd.get_dummies(
        df["possui_servico_de_internet"].map({"No": "sem_internet", "Fiber optic": "fibra_optica", "DSL": "dsl"}),
        prefix="possui_servico_de_internet",
    ).astype(int)
    metodo_pagamento_dummies = pd.get_dummies(
        df["metodo_pagamento"].map({
            "Mailed check": "mailed_check",
            "Electronic check": "electronic_check",
            "Bank transfer (automatic)": "bank_transfer_automatic",
            "Credit card (automatic)": "credit_card_automatic",
        }),
        prefix="metodo_pagamento",
    ).astype(int)
    df = pd.concat(
        [df.drop(columns=["possui_servico_de_internet", "metodo_pagamento"]), servico_de_internet_dummies, metodo_pagamento_dummies],
        axis=1,
    )

    # removendo clientes com 0 meses e cobranca vazia
    total_cobrado_numerico = pd.to_numeric(df["total_cobrado_durante_trimestre"], errors="coerce")
    sem_historico = (df["meses_como_cliente"] == 0) & (total_cobrado_numerico.isna() | (total_cobrado_numerico == 0))
    df = df[~sem_historico].copy()

    # addons e remocao do total cobrado (colinearidade)
    df["quantidade_de_addons"] = df[colunas_addons].sum(axis=1)
    df = df.drop(columns=["total_cobrado_durante_trimestre"])

    # duplicatas
    df = df.drop_duplicates()

    print(f"Registros finais: {df.shape[0]} | Colunas: {df.shape[1]}")
    print(df["target"].value_counts(normalize=True).round(4))
    return df

def dividir_treino_teste(df, test_size, seed):
    X = df.drop(columns=["target"])
    y = df["target"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=seed, stratify=y)

    print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
    return X, y, X_train, X_test, y_train, y_test

def criar_pipeline_mlp(colunas_para_escalonar, seed, **parametros_mlp):
    escalonador = ColumnTransformer(
        [("escala", StandardScaler(), colunas_para_escalonar)],
        remainder="passthrough",
        verbose_feature_names_out=False,
    )
    mlp = MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        alpha=1e-3,
        learning_rate_init=1e-3,
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=seed,
    )
    mlp.set_params(**parametros_mlp)
    return Pipeline([
        ("escala", escalonador),
        ("smote", SMOTE(random_state=seed)),
        ("mlp", mlp),
    ])

def validar_em_kfold(modelo, X, y, k_fold, metricas_cv):
    return cross_validate(modelo, X, y, cv=k_fold, scoring=metricas_cv)

def exibir_metricas_kfold(resultado_cv, metricas_cv):
    tabela_metricas_cv = pd.DataFrame({nome: resultado_cv[f"test_{nome}"] for nome in metricas_cv})
    tabela_metricas_cv.insert(0, "fold", range(1, 6))
    print(tabela_metricas_cv.round(4).to_string(index=False))
    print("\nMedia +/- desvio padrao entre os 5 folds:")
    for nome in metricas_cv:
        print(f"{nome}: {tabela_metricas_cv[nome].mean():.4f} +/- {tabela_metricas_cv[nome].std():.4f}")
    return tabela_metricas_cv

def buscar_hiperparametros(modelo, grade_de_parametros, X_train, y_train, k_fold):
    busca = GridSearchCV(
        modelo,
        grade_de_parametros,
        cv=k_fold,
        scoring="roc_auc",
        n_jobs=-1,
    )
    busca.fit(X_train, y_train)

    print("Melhores parametros:", busca.best_params_)
    print(f"AUC-ROC media na validacao cruzada: {busca.best_score_:.4f}")

    ranking = pd.DataFrame(busca.cv_results_)[["params", "mean_test_score", "std_test_score", "rank_test_score"]]
    return busca, ranking.sort_values("rank_test_score").head(8)

def resumir_kfold(resultado_cv, metricas_cv):
    cv_resumo = {nome: (resultado_cv[f"test_{nome}"].mean(), resultado_cv[f"test_{nome}"].std()) for nome in metricas_cv}
    for nome, (media, desvio) in cv_resumo.items():
        print(f"{nome}: {media:.4f} +/- {desvio:.4f}")
    return cv_resumo

def exibir_curva_de_aprendizado(pipeline):
    rede_treinada = pipeline.named_steps["mlp"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(rede_treinada.loss_curve_, color="coral")
    axes[0].set_title("Perda (loss) no treino por epoca")
    axes[0].set_xlabel("Epoca")
    axes[1].plot(rede_treinada.validation_scores_, color="steelblue")
    axes[1].set_title("Acuracia na validacao interna (early stopping)")
    axes[1].set_xlabel("Epoca")
    plt.tight_layout()
    plt.show()

    print(f"Epocas treinadas: {rede_treinada.n_iter_}")

def custo_de_negocio(y_real, y_previsto, cobranca_mensal, premissas_custo):
    y_real, y_previsto, cobranca_mensal = np.asarray(y_real), np.asarray(y_previsto), np.asarray(cobranca_mensal)
    falsos_negativos = (y_real == 1) & (y_previsto == 0)
    falsos_positivos = (y_real == 0) & (y_previsto == 1)
    custo_fn = (cobranca_mensal[falsos_negativos] * premissas_custo["meses_de_receita_perdida"]).sum()
    custo_fp = falsos_positivos.sum() * premissas_custo["custo_de_um_contato"]
    return custo_fn + custo_fp

def escolher_limiar_por_custo(modelo, X_train, y_train, k_fold, premissas_custo):
    proba_cv_treino = cross_val_predict(modelo, X_train, y_train, cv=k_fold, method="predict_proba")[:, 1]

    limiares = np.round(np.arange(0.05, 0.96, 0.01), 2)
    tabela_limiares = pd.DataFrame({
        "limiar": limiares,
        "custo": [custo_de_negocio(y_train, (proba_cv_treino >= t).astype(int), X_train["cobranca_mensal"], premissas_custo) for t in limiares],
        "f1_score": [f1_score(y_train, (proba_cv_treino >= t).astype(int)) for t in limiares],
    })
    limiar_escolhido = float(tabela_limiares.loc[tabela_limiares["custo"].idxmin(), "limiar"])
    print(f"Limiar de menor custo (validacao cruzada no treino): {limiar_escolhido:.2f}")
    return tabela_limiares, limiar_escolhido

def exibir_custo_por_limiar(tabela_limiares, limiar_escolhido):
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(tabela_limiares["limiar"], tabela_limiares["custo"], color="coral", label="Custo de negocio")
    ax1.axvline(limiar_escolhido, linestyle="--", color="gray")
    ax1.set_xlabel("Limiar")
    ax1.set_ylabel("Custo")
    ax2 = ax1.twinx()
    ax2.plot(tabela_limiares["limiar"], tabela_limiares["f1_score"], color="steelblue", label="F1")
    ax2.set_ylabel("F1")
    fig.legend(loc="lower center", ncol=2, frameon=False)
    plt.title("Custo de negocio e F1 por limiar")
    plt.tight_layout(rect=[0, 0.08, 1, 1])
    plt.show()

def metricas_no_teste(y_test, y_proba, cobranca_mensal, limiar, premissas_custo):
    y_pred = (y_proba >= limiar).astype(int)
    return {
        "limiar": limiar,
        "acuracia": accuracy_score(y_test, y_pred),
        "precisao": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "custo_de_negocio": custo_de_negocio(y_test, y_pred, cobranca_mensal, premissas_custo),
    }

def exibir_matriz_e_roc(y_test, y_proba, limiar):
    y_pred = (y_proba >= limiar).astype(int)
    matriz_confusao = confusion_matrix(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_proba)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ConfusionMatrixDisplay(confusion_matrix=matriz_confusao, display_labels=["Nao cancelou", "Cancelou"]).plot(ax=axes[0], cmap="Blues")
    axes[0].set_title(f"Matriz de Confusao (limiar {limiar:.2f})")
    axes[1].plot(fpr, tpr, color="coral", label=f"ROC AUC = {roc_auc_score(y_test, y_proba):.4f}")
    axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray")
    axes[1].set_xlabel("Taxa de Falsos Positivos")
    axes[1].set_ylabel("Taxa de Verdadeiros Positivos")
    axes[1].set_title("Curva ROC")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
    return matriz_confusao

def salvar_metricas(metricas, diretorio):
    diretorio = Path(diretorio)
    diretorio.mkdir(parents=True, exist_ok=True)
    with open(diretorio / "metrics.json", "w") as arquivo_metricas:
        json.dump(metricas, arquivo_metricas, indent=4, default=float)
    print("Metricas salvas em:", diretorio / "metrics.json")

def salvar_modelo(pipeline, limiar, colunas, diretorio):
    diretorio = Path(diretorio)
    diretorio.mkdir(parents=True, exist_ok=True)
    joblib.dump(
        {"pipeline": pipeline, "limiar": limiar, "colunas": colunas},
        diretorio / "mlp_classifier.joblib",
    )
    print("Modelo salvo em:", diretorio / "mlp_classifier.joblib")


## Carga dos dados

In [ ]:
raiz_do_projeto = encontrar_raiz_do_projeto(Path.cwd())

dataframe_de_churn_inicial = carregar_base(raiz_do_projeto / "data" / "raw" / "Telco-Customer-Churn.csv")
dataframe_de_churn_inicial.shape

## Tratamento dos dados (mesmo do V10)

In [ ]:
colunas_pt_br = {
    "customerID": "id_cliente",
    "gender": "genero",
    "SeniorCitizen": "cliente_em_idade_avancada",
    "Partner": "casado",
    "Dependents": "dependentes",
    "tenure": "meses_como_cliente",
    "PhoneService": "possui_servico_telefonico",
    "MultipleLines": "possui_multiplas_linhas",
    "InternetService": "possui_servico_de_internet",
    "OnlineSecurity": "possui_seguranca_online",
    "OnlineBackup": "possui_backup_online",
    "DeviceProtection": "possui_protecao_dispositivo",
    "TechSupport": "possui_suporte_tecnico_premium",
    "StreamingTV": "possui_streaming_tv_terceiros",
    "StreamingMovies": "possui_streaming_filmes_terceiros",
    "Contract": "tipo_contrato",
    "PaperlessBilling": "fatura_sem_papel",
    "PaymentMethod": "metodo_pagamento",
    "MonthlyCharges": "cobranca_mensal",
    "TotalCharges": "total_cobrado_durante_trimestre",
    "Churn": "target",
}

colunas_addons = [
    "possui_seguranca_online",
    "possui_backup_online",
    "possui_protecao_dispositivo",
    "possui_suporte_tecnico_premium",
    "possui_streaming_tv_terceiros",
    "possui_streaming_filmes_terceiros",
]

df = tratar_dados(dataframe_de_churn_inicial, colunas_pt_br, colunas_addons)

## Dividindo dados entre treino e teste

In [ ]:
X, y, X_train, X_test, y_train, y_test = dividir_treino_teste(df, test_size=0.3, seed=42)

## Persistindo a base final de treinamento em joblib (versionada no DVC)

In [ ]:
dados_processados_dir = raiz_do_projeto / "data" / "processed" / "redes_neurais"
dados_processados_dir.mkdir(parents=True, exist_ok=True)

caminho_base_treinamento = dados_processados_dir / "df.joblib"
joblib.dump(df, caminho_base_treinamento)

print(f"Base final de treinamento salva em: {caminho_base_treinamento}")
print(f"Shape: {df.shape}")

resultado_dvc_add = subprocess.run(
    ["dvc", "add", str(caminho_base_treinamento)],
    cwd=str(raiz_do_projeto),
    capture_output=True, text=True
)
print(resultado_dvc_add.stdout)
print(resultado_dvc_add.stderr)

## Pipeline com escalonamento, SMOTE e MLPClassifier

In [ ]:
colunas_para_escalonar = ["meses_como_cliente", "tipo_contrato", "cobranca_mensal", "quantidade_de_addons"]

modelo_mlp = criar_pipeline_mlp(colunas_para_escalonar, seed=42)
modelo_mlp

## Validacao cruzada estratificada (K-Fold) da rede inicial

In [ ]:
k_fold_estratificado = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metricas_cv = {"acuracia": "accuracy", "precisao": "precision", "recall": "recall", "f1_score": "f1", "roc_auc": "roc_auc"}

resultado_cv = validar_em_kfold(modelo_mlp, X_train, y_train, k_fold_estratificado, metricas_cv)

tabela_metricas_cv = exibir_metricas_kfold(resultado_cv, metricas_cv)

## Busca de hiperparametros com GridSearch

In [ ]:
grade_de_parametros = {
    "mlp__hidden_layer_sizes": [(16,), (32,), (32, 16), (64, 32)],
    "mlp__alpha": [1e-4, 1e-3, 1e-2],
    "mlp__learning_rate_init": [1e-3, 1e-2],
}
k_fold_estratificado = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
colunas_para_escalonar = ["meses_como_cliente", "tipo_contrato", "cobranca_mensal", "quantidade_de_addons"]

busca, ranking_busca = buscar_hiperparametros(
    criar_pipeline_mlp(colunas_para_escalonar, seed=42),
    grade_de_parametros, X_train, y_train, k_fold_estratificado,
)
ranking_busca

## Validacao cruzada da melhor rede

In [ ]:
k_fold_estratificado = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metricas_cv = {"acuracia": "accuracy", "precisao": "precision", "recall": "recall", "f1_score": "f1", "roc_auc": "roc_auc"}

melhor_mlp = busca.best_estimator_

resultado_cv_melhor = validar_em_kfold(melhor_mlp, X_train, y_train, k_fold_estratificado, metricas_cv)
cv_melhor = resumir_kfold(resultado_cv_melhor, metricas_cv)

## Curva de aprendizado

In [ ]:
exibir_curva_de_aprendizado(melhor_mlp)

## Escolha do limiar pelo custo de negocio
Custo = FN x (cobranca_mensal x meses perdidos) + FP x custo do contato. Premissas provisorias.

In [ ]:
premissas_custo = {
    "meses_de_receita_perdida": 12,   # premissa provisoria
    "custo_de_um_contato": 50.0,      # premissa provisoria
}
k_fold_estratificado = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

tabela_limiares, limiar_escolhido = escolher_limiar_por_custo(
    melhor_mlp, X_train, y_train, k_fold_estratificado, premissas_custo
)

exibir_custo_por_limiar(tabela_limiares, limiar_escolhido)

## Avaliacao no teste

In [ ]:
y_proba = melhor_mlp.predict_proba(X_test)[:, 1]

# limiar 0.5 (igual ao V10) e limiar de menor custo
resultado_teste = pd.DataFrame([
    metricas_no_teste(y_test, y_proba, X_test["cobranca_mensal"], 0.5, premissas_custo),
    metricas_no_teste(y_test, y_proba, X_test["cobranca_mensal"], limiar_escolhido, premissas_custo),
])
print(resultado_teste.round(4).to_string(index=False))

matriz_confusao = exibir_matriz_e_roc(y_test, y_proba, limiar_escolhido)

## Comparacao com a Regressao Logistica (V10)

In [ ]:
baseline_v10_teste = {"modelo": "Regressao Logistica (V10)", "limiar": 0.5, "acuracia": 0.7725, "precisao": 0.5557,
                      "recall": 0.7034, "f1_score": 0.6209, "roc_auc": 0.8308}

mlp_teste_05 = {"modelo": "MLPClassifier (V1)", **metricas_no_teste(y_test, y_proba, X_test["cobranca_mensal"], 0.5, premissas_custo)}
mlp_teste_custo = {"modelo": "MLPClassifier (V1) - limiar de custo", **metricas_no_teste(y_test, y_proba, X_test["cobranca_mensal"], limiar_escolhido, premissas_custo)}

tabela_comparativa = pd.DataFrame([baseline_v10_teste, mlp_teste_05, mlp_teste_custo])
tabela_comparativa.round(4)

## Registro de metricas e salvamento do modelo

In [ ]:
pasta_do_notebook = raiz_do_projeto / "notebooks" / "redes_neurais" / "historico"
metricas_dir = pasta_do_notebook / "metrics"

metricas = {
    "melhores_parametros": {k: (list(v) if isinstance(v, tuple) else v) for k, v in busca.best_params_.items()},
    "cv_treino": {nome: {"media": media, "desvio": desvio} for nome, (media, desvio) in cv_melhor.items()},
    "teste_limiar_0_5": metricas_no_teste(y_test, y_proba, X_test["cobranca_mensal"], 0.5, premissas_custo),
    "teste_limiar_custo": metricas_no_teste(y_test, y_proba, X_test["cobranca_mensal"], limiar_escolhido, premissas_custo),
    "premissas_custo": premissas_custo,
    "matriz_confusao_limiar_custo": matriz_confusao.tolist(),
}
salvar_metricas(metricas, metricas_dir)

pasta_modelos = raiz_do_projeto / "models" / "v1.0.0"
salvar_modelo(melhor_mlp, limiar_escolhido, X.columns.tolist(), pasta_modelos)